In [1]:
import torch
import pandas as pd
import numpy as np
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import scanpy as sc
import sys 
task_grn_inference_dir = '../task_grn_inference'
sys.path.append(task_grn_inference_dir)
from src.utils.util import efficient_melting
torch.cuda.is_available()

False

In [68]:
par = {
    'prior_net': 'output/grns/pearson_corr.csv',
    'train_adata': f'{task_grn_inference_dir}/resources/inference_datasets/op_rna.h5ad'
}

prior_net = pd.read_csv(par['prior_net'])
prior_net['weight'] = prior_net['weight'].abs()

# Load Single-Cell Data
train_adata = sc.read_h5ad(par['train_adata'])  

all_nodes = list(set(prior_net['source']) | set(prior_net['target']))
node_map = {node: idx for idx, node in enumerate(all_nodes)}


# - adjust adata to net to have the same nodes #TODO: fix this to make net compatible with adata
train_adata = train_adata[:, train_adata.var_names.isin(all_nodes)]
gene_list = train_adata.var_names.tolist()
X = train_adata.X.todense().A



# Map edges to indices
edges = prior_net[['source', 'target']].applymap(node_map.get).values.T #2*50000
edge_weights = torch.tensor(prior_net['weight'].values, dtype=torch.float32) #50000

# Create node features (expression data)
n_nodes = len(all_nodes)
n_features =  X.shape[0]
node_features = torch.zeros((n_nodes, n_features)) 

for node, idx in node_map.items():
    if node in gene_list:
        gene_idx = gene_list.index(node)
        node_features[idx] = torch.tensor(X[:, gene_idx].flatten())

/vol/tmp/users/jnourisa/ipykernel_2644960/4086140035.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  edges = prior_net[['source', 'target']].applymap(node_map.get).values.T #2*50000


In [69]:
# --- Step 3: Define GNN ---

# Create PyG Data object
graph_data = Data(
    x=node_features, 
    edge_index=torch.tensor(edges, dtype=torch.long),
    edge_attr=edge_weights
)

In [75]:
class GNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

        # Apply Xavier initialization to both GCNConv layers
        self.reset_parameters()

    def reset_parameters(self):
        # Apply Xavier initialization to the weights
        torch.nn.init.xavier_uniform_(self.conv1.lin.weight)
        torch.nn.init.xavier_uniform_(self.conv2.lin.weight)
        if self.conv1.lin.bias is not None:
            self.conv1.lin.bias.data.fill_(0)  # Initialize biases to zero
        if self.conv2.lin.bias is not None:
            self.conv2.lin.bias.data.fill_(0)

    def forward(self, x, edge_index, edge_weight):
        # GNN forward pass
        x_hidden = self.conv1(x, edge_index, edge_weight=edge_weight)
        # x_hidden = torch.relu(x_hidden)
        x_hidden = torch.nn.functional.leaky_relu(x_hidden)
        x = self.conv2(x_hidden, edge_index, edge_weight=edge_weight)
        return x, x_hidden

# --- Step 4: Train the GNN ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
graph_data = graph_data.to(device)

model = GNN(in_channels=X.shape[0], hidden_channels=64, out_channels=n_features).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = torch.nn.MSELoss()  # Change based on your task

# x_latent = np.nan
for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    out, x_hidden = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
    mask = graph_data.x != 0
    loss = loss_fn(out[mask], graph_data.x[mask])
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

Epoch 1, Loss: 98.80047607421875
Epoch 2, Loss: 9624.5859375
Epoch 3, Loss: 7694.587890625
Epoch 4, Loss: 4267.595703125
Epoch 5, Loss: 417.2795104980469
Epoch 6, Loss: 2193.45849609375
Epoch 7, Loss: 805.4013061523438
Epoch 8, Loss: 269.8671875
Epoch 9, Loss: 231.41482543945312
Epoch 10, Loss: 134.8854217529297
Epoch 11, Loss: 87.56732940673828
Epoch 12, Loss: 91.47964477539062
Epoch 13, Loss: 99.53947448730469
Epoch 14, Loss: 106.94901275634766
Epoch 15, Loss: 107.38239288330078
Epoch 16, Loss: 99.2058334350586
Epoch 17, Loss: 90.17887115478516
Epoch 18, Loss: 78.36797332763672
Epoch 19, Loss: 73.67343139648438
Epoch 20, Loss: 79.10662841796875
Epoch 21, Loss: 91.96884155273438
Epoch 22, Loss: 87.08824157714844
Epoch 23, Loss: 75.89067077636719
Epoch 24, Loss: 83.43888854980469
Epoch 25, Loss: 80.52935028076172
Epoch 26, Loss: 67.53901672363281
Epoch 27, Loss: 67.80506134033203
Epoch 28, Loss: 70.6767807006836
Epoch 29, Loss: 64.51370239257812
Epoch 30, Loss: 64.07403564453125
Epoch 

KeyboardInterrupt: 

In [82]:
x_latent.std(1)

array([ 6.2163863 ,  0.05056819,  9.394516  , ...,  6.251745  ,
       30.61748   ,  7.315578  ], dtype=float32)

In [77]:
x_latent = x_hidden.detach().numpy()

corr = np.corrcoef(x_latent)

efficient_melting(corr, gene_list, {'max_n_links': 50000})

convert to long dataframe


,target,source,weight
0,SKI,HES4,0.9998704311126189
1,PRDM16,HES4,0.9999113852634497
2,TP73,HES4,0.9999593649516072
3,HES2,HES4,0.9991619421211334
4,ENO1,HES4,0.9999795478457403
...,...,...,...
6590260,MT-ND5,MT-ND4L,0.9999611107711643
6590261,MT-CYB,MT-ND4L,0.9994277276371738
6590262,MT-ND5,MT-ND4,0.994925578969881
6590263,MT-CYB,MT-ND4,0.9972106006424898


In [16]:
import torch

# Node embeddings (output from the GNN)
node_embeddings = out  # Shape: (num_nodes, hidden_dim)

# Extract edge indices
edge_index = graph_data.edge_index  # Shape: (2, num_edges)

# Extract embeddings for source and target nodes
source_embeddings = node_embeddings[edge_index[0]]  # Shape: (num_edges, hidden_dim)
target_embeddings = node_embeddings[edge_index[1]]  # Shape: (num_edges, hidden_dim)

# Compute new edge weights (cosine similarity, dot product, or other metric)
# Example: Cosine similarity
cosine_similarity = torch.nn.functional.cosine_similarity(source_embeddings, target_embeddings, dim=-1)

# Example: Dot product
dot_product = torch.sum(source_embeddings * target_embeddings, dim=-1)  # Shape: (num_edges,)

# Use the desired metric as the updated edge weights
new_edge_weights = cosine_similarity  # or dot_product
new_edge_weights

tensor([1., 1., 1.,  ..., 1., 1., 1.], grad_fn=<SumBackward1>)

# Benchmark

In [ ]:
par = {
    'grn_models': [
        'pearson_corr', 
        # 'grnboost2', 'celloracle', 'scenicplus',
        'net_all_celltypes_all_ages_all_batches', 
        'net_all_celltypes_young_all_batches',
        'net_all_celltypes_young_batch_1',
        'net_all_celltypes_old_all_batches',
        'net_all_celltypes_old_batch_1', 
        'net_B cells_all_ages_all_batches', 'net_T cells_all_ages_all_batches', 'net_Myeloid cells_all_ages_all_batches', 'net_NK cells_all_ages_all_batches',
        ],
    'grn_models_dir':'output/grns/'
}
if True: # topology
    from src.helper import topology_stats
    topology_stats(par)


In [ ]:


df_scores = pd.read_csv(f"output/scores/X_norm-50000-skeleton_False-binarize_False-ridge-global-False.csv", index_col=0)

df_scores = df_scores.reindex(par['grn_models'])
# df_scores = df_scores.fillna(0)
df_scores.columns = df_scores.columns.map(surragate_names)
df_scores.index = df_scores.index.map({**surragate_names, ** {'net_all_celltypes_all_ages_all_batches':'PBMC All', 
                                       'net_all_celltypes_young_all_batches': 'PBMC Young', 
                                       'net_all_celltypes_young_batch_1':'PBMC Young batch1',
                                       'net_all_celltypes_old_all_batches': 'PBMC Old',
                                       'net_all_celltypes_old_batch_1': 'PBMC Old batch1',
                                       'net_B cells_all_ages_all_batches': 'PBMC B cells',
                                       'net_T cells_all_ages_all_batches': 'PBMC T cells',
                                       'net_Myeloid cells_all_ages_all_batches': 'PBMC Myeloid cells',
                                       'net_NK cells_all_ages_all_batches': 'PBMC NK cells'}})

# df_scores_gb = pd.read_csv(f"{task_grn_inference_dir}/resources/scores/op/X_norm-50000-skeleton_False-binarize_True-ridge-global-True.csv", index_col=0)
# df_scores_gb.columns = df_scores_gb.columns.map(surragate_names)

# df_scores = pd.concat([df_scores, df_scores_gb])

fig, ax = plt.subplots(1, 1, figsize=(3, 6), sharey=False, dpi=100)

plot_heatmap(df_scores, name='', ax=ax, cmap="viridis", fmt='0.03f')